# Experiment 2 — Dataset Sensitivity Analysis Across 6 Cases

---

## Overview

This notebook is **Experiment 2** and builds directly on the Genetic Algorithm (GA) framework established in `Meal_Plan_Optimization_GA.ipynb` (Experiment 1). Readers unfamiliar with the algorithm design — chromosome representation, fitness function, crossover, mutation, selection, or the three baseline algorithms (RS, LS, SA) — should refer to Experiment 1 first.

### What Is New in This Experiment

Experiment 1 evaluated the GA on a single, fixed food dataset (`Processed_Bahrain_Food_Dataset.csv`) across all 10 calorie targets with two weight configurations.

**Experiment 2** was done to check if any change to the dataset would affect the performance

To do this, the same GA and baselines are run on **6 carefully constructed dataset variants**, differing in:

| Dimension | Values Tested |
|---|---|
| Dataset size | 1,000 rows vs. 5,000 rows |
| Augmentation method | Original + Synthetic, Original + Duplicates, Original + Synthetic + Duplicates |

This produces a **2 × 3 factorial design** of 6 cases.

### Calorie Targets

To keep runtime manageable across 6 cases × 4 algorithms × 10 runs, three representative calorie targets are used: **1,200 / 1,600 / 2,200 kcal**.

### Dataset Cases

| Case | Size | Augmentation |
|---|---|---|
| Case 1 | 1,000 rows | Original + Synthetic |
| Case 2 | 1,000 rows | Original + Duplicates |
| Case 3 | 5,000 rows | Original + Synthetic |
| Case 4 | 5,000 rows | Original + Duplicates |
| Case 5 | 1,000 rows | Original + Synthetic + Duplicates |
| Case 6 | 5,000 rows | Original + Synthetic + Duplicates |

### Outputs

- Per-case fitness tables (mean, max, min, std) across all algorithms
- Combined box plots and grouped bar charts across all 6 cases
- GA runtime comparison per case
- Wilcoxon signed-rank statistical tests (GA vs. each baseline)
- Exported CSV summaries and heatmap of p-values

---




## 1. Import Libraries

Same libraries as Experiment 1, with the addition of **`pandas`**, which is used in this experiment for building the combined results summary table and exporting it to CSV.


In [4]:
# Imports
from dataclasses import dataclass
import random
import numpy as np
import csv
from deap.tools import cxOnePoint, cxTwoPoints
import time
import copy
import random
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import wilcoxon
from deap import base, creator, tools, algorithms

random.seed(42)



## 2. Data Loading and Preprocessing

### 2.1 Gene and Chromosome Representation

Each **Gene** stores one food item and its serving count for a specific (meal, food group) slot. A full **Chromosome** contains 30 genes: 5 meals x 6 food groups.


In [5]:
@dataclass
class Gene:
    food_item_name: str
    serving: int

### 2.2 Meal and Food Group Structure

In [6]:
# A list representing all the meals we have in 1 chromosome
Meals = ["Breakfast", "Snack 1", "Lunch", "Snack 2", "Dinner"]
# A list representing all the food groups we have in 1 meal
Food_Groups = ["Vegetables", "Fruits", "Grains", "Protein", "Dairy", "Fats and Oils"]

### 2.3 Dataset Loader

In Experiment 2 this function is called **once per case** with a different CSV path, allowing each of the 6 dataset variants to be loaded independently without restarting the kernel.


In [7]:
def load_food_data(csv_path):
    """
    Parses a food dataset CSV into two dictionaries:
      food_data  -- {food group -> {food item -> nutritional stats}}
      meal_flags -- {food item -> {meal -> 0 or 1 suitability flag}}

    Missing or non-numeric values are safely defaulted to 0.
    """
    food_data  = {}
    meal_flags = {}

    with open(csv_path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)

        for row in reader:
            group = row['Category']
            food  = row['Food Item']

            if group not in food_data:
                food_data[group] = {}

            # Helper: convert to float, return 0.0 if the value is missing or invalid
            def safe_float(x):
                try:
                    return float(x)
                except (ValueError, TypeError):
                    return 0.0

            # Helper: convert to int, return 0 if the value is missing or invalid
            def safe_int(x):
                try:
                    return int(x)
                except (ValueError, TypeError):
                    return 0

            # Store macronutrient data and user preference for each food item
            food_data[group][food] = {
                'p':    safe_float(row['Protein (g)']),
                'f':    safe_float(row['Fats (g)']),
                'c':    safe_float(row['Carbohydrates (g)']),
                'cal':  safe_float(row['Calories (kcal)']),
                'pref': safe_float(row['User Preference'])
            }

            # Store which meals this food item is allowed in (1 = allowed, 0 = not)
            meal_flags[food] = {
                'Breakfast': safe_int(row['Breakfast']),
                'Snack 1':   safe_int(row['Snack 1']),
                'Lunch':     safe_int(row['Lunch']),
                'Snack 2':   safe_int(row['Snack 2']),
                'Dinner':    safe_int(row['Dinner'])
            }

    return food_data, meal_flags


# Initial load of the base dataset.
# Note: run_case() will reload FOOD_DATA and MEAL_FLAGS with its own CSV
# before each case runs, so this is only used if you call evaluate_meal_plan()
# or create_individual() outside of a case (e.g. for quick testing).
FOOD_DATA, MEAL_FLAGS = load_food_data("..\\Datasets\\Processed_Bahrain_Food_Dataset.csv")

FileNotFoundError: [Errno 2] No such file or directory: '..\\Datasets\\Processed_Bahrain_Food_Dataset.csv'

### 2.4 Saudi Dietary Serving Guidelines

Serving counts per food group per meal for each of the 10 calorie targets. These act as hard structural constraints -- every chromosome is built to match them exactly, so only food item selection evolves.


In [ ]:
# Official Saudi dietary guidelines: for each calorie target, how many servings
# of each food group should appear at each meal. Serving counts are fixed;
# only the food item chosen for each slot is optimized by the GA.
SERVING_GUIDELINES_BY_KCAL = {
    1200: {
        "Breakfast": {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1": {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch": {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 2, "Fruits": 0, "Fats and Oils": 0},
        "Snack 2": {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner": {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 0}
    },

    1300: {
        "Breakfast": {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1": {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch": {"Grains": 3, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2": {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner": {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 0}
    },

    1400: {
        "Breakfast": {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1": {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch": {"Grains": 3, "Dairy": 1, "Protein": 1, "Vegetables": 2, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner": {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 0}
    },

    1500: {
        "Breakfast": {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1": {"Grains": 0, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch": {"Grains": 3, "Dairy": 1, "Protein": 2, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner": {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 0}
    },

    1600: {
        "Breakfast": {"Grains": 2, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch": {"Grains": 3, "Dairy": 1, "Protein": 2, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner": {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 0}
    },

    1700: {
        "Breakfast": {"Grains": 3, "Dairy": 1, "Protein": 1, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch": {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 2, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner": {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 0}
    },

    1800: {
        "Breakfast": {"Grains": 3, "Dairy": 1, "Protein": 2, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch": {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner": {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 1}
    },

    1900: {
        "Breakfast": {"Grains": 3, "Dairy": 1, "Protein": 2, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch": {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 2, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner": {"Grains": 2, "Dairy": 0, "Protein": 1, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 1}
    },

    2000: {
        "Breakfast": {"Grains": 3, "Dairy": 1, "Protein": 2, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch": {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2": {"Grains": 2, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner": {"Grains": 2, "Dairy": 0, "Protein": 2, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 0}
    },

    2200: {
        "Breakfast": {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 1, "Fruits": 0, "Fats and Oils": 1},
        "Snack 1": {"Grains": 1, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Lunch": {"Grains": 3, "Dairy": 1, "Protein": 3, "Vegetables": 2, "Fruits": 0, "Fats and Oils": 1},
        "Snack 2": {"Grains": 2, "Dairy": 0, "Protein": 0, "Vegetables": 0, "Fruits": 1, "Fats and Oils": 0},
        "Dinner": {"Grains": 2, "Dairy": 0, "Protein": 2, "Vegetables": 1, "Fruits": 1, "Fats and Oils": 1}
    }
}



## 3. Genetic Algorithm Implementation

### 3.1 DEAP Configuration and Population Initialization

DEAP is configured with a maximization objective. The `create_individual` function builds a chromosome by reading serving counts from the Saudi guidelines for `TARGET_KCAL` and randomly picking a meal-compatible food item for each of the 30 slots.


In [ ]:
# Register a maximization fitness (weight=+1.0) and an Individual type that wraps a list
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()


def create_individual():
    """
    Builds one chromosome: 30 genes covering 5 meals x 6 food groups.
    Serving counts are fixed by the Saudi guidelines for TARGET_KCAL;
    only the food item for each slot is chosen randomly from meal-compatible options.
    """
    genes = []
    for meal in Meals:
        for group in Food_Groups:
            # Look up how many servings this food group requires at this meal
            serving_count = SERVING_GUIDELINES_BY_KCAL[TARGET_KCAL][meal][group]

            # Only consider foods that are flagged as suitable for this meal
            allowed_foods = [
                food for food in FOOD_DATA[group].keys()
                if MEAL_FLAGS[food][meal] == 1
            ]
            # Fallback: if no food is flagged for this meal, allow any food in the group
            if not allowed_foods:
                allowed_foods = list(FOOD_DATA[group].keys())

            food_name = random.choice(allowed_foods)
            genes.append(Gene(food_item_name=food_name, serving=serving_count))

    return creator.Individual(genes)


toolbox.register("individual", create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)


### 3.2 Fitness Function

Four objectives are combined into a single weighted score. Because `score` is minimized internally, fitness is returned as `1 / (1 + score)` so DEAP can maximize it (range: 0-1, higher = better).

> **Note:** In Experiment 2 the weights are fixed at equal weighting (0.25 each), reflecting the balanced multi-objective design used in this sensitivity study.


In [ ]:
def evaluate_meal_plan(individual):
    """
    Evaluates a chromosome against four nutritional objectives:
      j_macro -- squared error between actual and AMDR target macro ratios (55/20/25%)
      j_cal   -- squared relative error between total calories and TARGET_KCAL
      j_var   -- penalty for repeated food items across the plan (variety)
      j_pref  -- average user preference penalty (0=most liked, 1=most disliked)

    Returns a single-element tuple (fitness,) as required by DEAP.
    """
    total_p    = 0.0  # total protein grams across all meals
    total_f    = 0.0  # total fat grams
    total_c    = 0.0  # total carbohydrate grams
    total_cal  = 0.0  # total calories
    total_pref = 0.0  # sum of preference penalties for included food items

    # Accumulate nutritional totals by looking up each gene's food item
    for gene in individual:
        group_name = None
        for group, foods in FOOD_DATA.items():
            if gene.food_item_name in foods:
                group_name = group
                break

        if group_name:
            stats      = FOOD_DATA[group_name][gene.food_item_name]
            total_p   += stats['p']   * gene.serving
            total_f   += stats['f']   * gene.serving
            total_c   += stats['c']   * gene.serving
            total_cal += stats['cal'] * gene.serving
            if gene.serving > 0:
                total_pref += stats['pref']

    g_total = total_p + total_f + total_c
    if g_total == 0:
        return 0,  # skip evaluation if the plan has no macronutrient content

    # Actual macro proportions
    p_p = total_p / g_total
    p_f = total_f / g_total
    p_c = total_c / g_total

    # AMDR target ratios: 55% carbs, 20% protein, 25% fats
    r_p, r_f, r_c = 0.20, 0.25, 0.55

    # Objective 1: macronutrient balance -- penalise deviation from AMDR targets
    j_macro = (p_c - r_c) ** 2 + (p_p - r_p) ** 2 + (p_f - r_f) ** 2

    # Objective 2: calorie accuracy -- penalise deviation from the daily calorie target
    r     = (total_cal - TARGET_KCAL) / TARGET_KCAL
    j_cal = min(1, r ** 2)

    # Objective 3: meal variety -- penalise any food item that appears more than once
    N = sum(1 for gene in individual if gene.serving > 0)
    food_appearance = {}
    for gene in individual:
        if gene.serving > 0:
            food_appearance[gene.food_item_name] = food_appearance.get(gene.food_item_name, 0) + 1

    j_var = 0
    for food_name, count in food_appearance.items():
        if count > 1:
            j_var += max(0, count - 1) ** 2
    j_var = j_var / ((N - 1) ** 2)  # normalize so the penalty is scale-independent

    # Objective 4: user preference -- lower is better (0 = liked, 1 = disliked)
    j_pref = total_pref / N

    # Equal weighting across all four objectives in Experiment 2
    w_macro, w_cal, w_var, w_pref = 0.25, 0.25, 0.25, 0.25
    score = w_macro * j_macro + w_var * j_var + w_cal * j_cal + w_pref * j_pref

    # Cache intermediate penalties on the individual for post-run inspection
    individual.j_var   = j_var
    individual.j_macro = j_macro
    individual.j_cal   = j_cal
    individual.j_pref  = j_pref

    # Convert minimization score to maximization fitness
    return 1 / (1 + score),


### 3.3 Crossover Operators

Two meal-boundary crossover operators are defined. Cuts always fall between complete meals, preserving the structural integrity of each meal's 6-gene block.


In [ ]:
def one_cx_meal_plan(parent1, parent2):
    """One-point crossover: swap all genes from a random meal boundary onward."""
    GROUPS = 6
    MEALS  = 5
    cut = random.randrange(1, MEALS) * GROUPS
    parent1[cut:], parent2[cut:] = parent2[cut:], parent1[cut:]
    return parent1, parent2


def two_cx_meal_plan(parent1, parent2):
    """Two-point crossover: swap the segment of meals between two random boundaries."""
    GROUPS = 6
    MEALS  = 5
    m1, m2     = sorted(random.sample(range(1, MEALS), 2))
    cut1, cut2 = m1 * GROUPS, m2 * GROUPS
    parent1[cut1:cut2], parent2[cut1:cut2] = parent2[cut1:cut2], parent1[cut1:cut2]
    return parent1, parent2


### 3.4 Mutation Operator

Replaces the food item of up to 2 randomly selected genes with a new meal-compatible alternative. Serving counts are never changed.


In [ ]:
def mutate_meal_plan(individual):
    """
    Mutates up to 2 genes by replacing their food item with a randomly chosen
    alternative from the same food group that is compatible with that meal.
    Serving counts remain fixed to the dietary guidelines.
    """
    mutations_done = 0
    all_indices    = list(range(len(individual)))
    random.shuffle(all_indices)  # randomise order so we don't always mutate the same genes

    for i in all_indices:
        if mutations_done >= 2:
            break  # cap at 2 mutations per call to avoid disrupting too much of the plan

        if individual[i].serving > 0:
            group_name    = Food_Groups[i % 6]  # food group determined by position
            meal          = Meals[i // 6]        # meal determined by position
            allowed_foods = [
                food for food in FOOD_DATA[group_name].keys()
                if MEAL_FLAGS[food][meal] == 1
            ]
            if allowed_foods:
                individual[i].food_item_name = random.choice(allowed_foods)
                mutations_done += 1

    return individual,


### 3.5 Selection and Operator Registration


In [ ]:
# Register all genetic operators in the DEAP toolbox
toolbox.register("evaluate", evaluate_meal_plan)
toolbox.register("select",   tools.selTournament, tournsize=3)  # 3 candidates compete per selection
toolbox.register("mate",     cxOnePoint)                        # DEAP built-in one-point crossover
toolbox.register("mutate",   mutate_meal_plan)


## 4. GA Execution

The `run_simulation` function runs one complete GA using DEAP's **(mu + lambda)** strategy with **early stopping**: if the best fitness does not improve by more than `EPS = 1e-6` for `PATIENCE` consecutive generations, evolution terminates early to avoid wasted computation.


In [ ]:
def run_simulation(target_kcal, pop_size, cxpb, mutpb, patience):
    """
    Runs one full GA for the given calorie target.

    Parameters
    ----------
    target_kcal : int   -- daily calorie target; sets TARGET_KCAL globally
    pop_size    : int   -- number of individuals in the population (mu)
    cxpb        : float -- crossover probability per offspring pair
    mutpb       : float -- mutation probability per offspring
    patience    : int   -- generations without improvement before early stopping

    Returns
    -------
    best_fitness : float -- highest fitness achieved across all generations
    stopping_gen : int   -- generation at which evolution terminated
    """
    global TARGET_KCAL
    TARGET_KCAL = target_kcal  # update global so create_individual uses the right guidelines

    pop = toolbox.population(n=pop_size)
    hof = tools.HallOfFame(1)  # always keeps the single best individual ever seen

    mu    = len(pop)  # mu: individuals selected into the next generation
    lambd = 100       # lambda: offspring produced each generation

    MAX_GEN  = 300
    PATIENCE = patience
    EPS      = 1e-6   # minimum improvement threshold -- smaller changes count as stagnation

    best_so_far = None
    stagnant    = 0

    stats = tools.Statistics(lambda ind: ind.fitness.values[0])
    stats.register("avg", np.mean)
    stats.register("min", np.min)
    stats.register("max", np.max)

    for gen in range(MAX_GEN):
        # Evolve for exactly one generation using the (mu + lambda) strategy
        algorithms.eaMuPlusLambda(
            pop, toolbox,
            mu=mu, lambda_=lambd,
            cxpb=cxpb, mutpb=mutpb,
            ngen=1, stats=stats, halloffame=hof, verbose=False
        )
        hof.update(pop)

        current_best = hof[0].fitness.values[0]

        # Check for meaningful improvement; reset or increment stagnation counter
        if best_so_far is None or current_best > best_so_far + EPS:
            best_so_far = current_best
            stagnant    = 0
        else:
            stagnant += 1

        # Stop early if fitness has not improved for PATIENCE consecutive generations
        if stagnant >= PATIENCE:
            break

    return hof[0].fitness.values[0], gen + 1




## 5. Comparative Algorithms

All three baselines run for the **same wall-clock time** as the GA run they are paired with, ensuring a fair time-controlled comparison.

### 5.1 Random Search


In [ ]:

def run_random_search(time_budget):
    """
    Repeatedly generates random individuals until the time budget is exhausted.
    Keeps track of the best individual found.
    Serves as the lower-bound baseline -- no learning or directed search.
    """
    start_time = time.time()

    best_individual = None
    best_fitness = -float("inf")
    trials = 0

    while (time.time() - start_time) < time_budget:
        ind = create_individual()
        fitness = evaluate_meal_plan(ind)[0]
        trials += 1

        if fitness > best_fitness:
            best_fitness = fitness
            best_individual = ind

    end_time = time.time()
    total_time = end_time - start_time

    return best_individual, best_fitness, trials, total_time

### 5.2 Local Search


In [ ]:
def run_local_search(toolbox, pop_size, time_budget):
    """
    Starts from one random individual in an initial population
    and iteratively accepts neighbours only if they improve fitness.
    Fast to exploit a good starting point but susceptible to local optima.
    """
    start_time = time.time()

    # Generate initial population
    pop = toolbox.population(n=pop_size)

    # Evaluate population
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)

    # Pick one random from initial population
    current = toolbox.clone(random.choice(pop))

    trials = 0

    # Local search loop
    while (time.time() - start_time) < time_budget:

        neighbor = toolbox.clone(current)
        neighbor, = mutate_meal_plan(neighbor)

        neighbor.fitness.values = toolbox.evaluate(neighbor)
        trials += 1

        if neighbor.fitness.values[0] > current.fitness.values[0]:
            current = toolbox.clone(neighbor)

    return current, current.fitness.values[0], trials

### 5.3 Simulated Annealing

In [ ]:

def fitness_to_cost(fitness):
    """Converts a maximization fitness back to an equivalent minimization cost."""
    return (1 / fitness) - 1


def get_neighbor(individual):
    """
    Returns a deep-copied neighbour by mutating up to 2 food item selections.
    Deep copy is used to avoid modifying the original individual in place.
    """
    new_individual = copy.deepcopy(individual)

    mutations_done = 0
    all_indices = list(range(len(new_individual)))
    random.shuffle(all_indices)

    for i in all_indices:
        if mutations_done >= 2:
            break

        if new_individual[i].serving > 0:
            group_name = Food_Groups[i % 6]
            meal = Meals[i // 6]

            allowed_foods = [
                food for food in FOOD_DATA[group_name].keys()
                if MEAL_FLAGS[food][meal] == 1
            ]
            if allowed_foods:
                new_individual[i].food_item_name = random.choice(allowed_foods)
                mutations_done += 1

    return new_individual


def run_simulated_annealing(toolbox, pop_size,
                            cooling_rate=0.98,
                            iterations_per_temp=200,
                            time_limit=None):
    """
    Simulated Annealing: accepts worse solutions with probability exp(-delta/T),
    where T decreases geometrically each step. This allows escape from local
    optima early on, converging to better solutions as temperature cools.

    Parameters
    ----------
    cooling_rate        : geometric factor by which T is multiplied each step
    iterations_per_temp : neighbour evaluations performed at each temperature level
    time_limit          : wall-clock seconds before the search is stopped
    """

    # Generate population
    current = toolbox.individual()
    current_fitness = evaluate_meal_plan(current)[0]
    current_cost = fitness_to_cost(current_fitness)


    best = copy.deepcopy(current)
    best_fitness = current_fitness

    T = 1.0
    min_temp=0.001
    total_iterations = 0
    start_time = time.time()


    while T > min_temp:
        if time_limit and (time.time() - start_time) >= time_limit:
            break
        for _ in range(iterations_per_temp):

            new = get_neighbor(current)
            new_fitness = evaluate_meal_plan(new)[0]
            new_cost = fitness_to_cost(new_fitness)

            delta = new_cost - current_cost

            if delta <= 0 or random.random() < math.exp(-delta / T):
                current = new
                current_fitness = new_fitness
                current_cost = new_cost

            if current_fitness > best_fitness:
                best = copy.deepcopy(current)
                best_fitness = current_fitness

            total_iterations += 1

        T *= cooling_rate

    return best, best_fitness, total_iterations

## 6. Meal Plan Display Utility

Prints a formatted nutritional summary for any evaluated chromosome.


In [ ]:
def print_meal_plan(individual):
    """Prints a structured daily nutrition summary for the given chromosome."""
    total_p = total_f = total_c = total_cal = 0.0

    for i, gene in enumerate(individual):
        meal = Meals[i // 6]
        group = Food_Groups[i % 6]
        stats = FOOD_DATA[group][gene.food_item_name]
        p = stats['p'] * gene.serving
        f = stats['f'] * gene.serving
        c = stats['c'] * gene.serving
        cal = stats['cal'] * gene.serving
        pref = stats['pref'] if gene.serving > 0 else 0
        total_p += p
        total_f += f
        total_c += c
        total_cal += cal


    g_total = total_p + total_f + total_c
    p_p = (total_p / g_total) * 100
    p_f = (total_f / g_total) * 100
    p_c = (total_c / g_total) * 100
    variety = (1 - individual.j_var) * 100
    calories = (total_cal / TARGET_KCAL) * 100
    preference_score = (1 - individual.j_pref) * 100

    print("\n" + "=" * 40)
    print("       DAILY NUTRITION SUMMARY")
    print("=" * 40)
    print(f"Total Energy:   {total_cal:.1f} kcal")
    print(f"Total Weight:   {g_total:.1f} g (Macro Grams)")
    print("-" * 40)
    print(f"Nutrient        | Actual % | Target %")
    print(f"Carbs (c)       | {p_c:>7.1f}% | 55.0%")
    print(f"Protein (p)     | {p_p:>7.1f}% | 20.0%")
    print(f"Fats (f)        | {p_f:>7.1f}% | 25.0%")
    print(f"Meal variety    | {variety:>7.1f}% | 100%")
    print(f"Meal calorie    | {calories:>7.1f}% | 100%")
    print(f"User preference | {preference_score:>7.1f}% | 100%")
    print("=" * 40)



## 7. Experiment 2 Setup -- Shared Configuration and `run_case`

### Important: Run This Cell Once Before Any Case Cell

This cell defines all experiment-wide parameters and the `run_case` function that drives every case. Results accumulate in `all_case_results` and `all_case_times`, which are consumed by the Combined Report and Wilcoxon cells.

**How `run_case` works:**
1. Loads the case-specific CSV into `FOOD_DATA` and `MEAL_FLAGS`.
2. Runs the GA 10 times per calorie target, recording fitness and runtime.
3. Uses the GA's **average runtime** as the time budget for RS, LS, and SA -- guaranteeing a fair, time-controlled comparison.
4. Stores all results under the case label key for later reporting.


In [ ]:
# ---- Experiment-wide parameters -- change here to affect all cases ----
EXP_TARGETS = [1200, 1600, 2200]   # representative calorie targets for this experiment
N_RUNS      = 10                   # independent runs per algorithm per target
POP_SIZE    = 95                   # GA population size (same as Experiment 1)
CXPB, MUTPB = 0.55, 0.17          # crossover and mutation probabilities
PATIENCE    = 50                   # early stopping patience (generations)
ALGOS       = ['GA', 'RS', 'LS', 'SA']
ALGO_COLORS = {'GA': '#4C72B0', 'RS': '#DD8452', 'LS': '#55A868', 'SA': '#C44E52'}

# Shared result stores -- each case cell appends its results here
all_case_results = {}   # {case_label: {target: {algo: {'fitness': [...], 'stats': {...}}}}}
all_case_times   = {}   # {case_label: {target: avg_ga_runtime_seconds}}


def _stats(lst):
    """Returns a dict of summary statistics for a list of fitness values."""
    return {
        'mean': float(np.mean(lst)),
        'max':  float(np.max(lst)),
        'min':  float(np.min(lst)),
        'std':  float(np.std(lst)),
    }


def run_case(case_label, csv_path):
    """
    Runs the full experiment for one dataset case:
      1. Loads the dataset from csv_path
      2. For each target in EXP_TARGETS, runs GA x N_RUNS and records fitness + runtime
      3. Uses mean GA runtime as the time budget for RS, LS, and SA
      4. Stores all results in all_case_results[case_label]

    Parameters
    ----------
    case_label : str -- human-readable name for this case (used as the results key)
    csv_path   : str -- path to the case-specific food dataset CSV
    """
    global TARGET_KCAL, FOOD_DATA, MEAL_FLAGS

    print(f"\n{'='*70}")
    print(f"  {case_label}")
    print(f"  Dataset : {csv_path}")
    print(f"{'='*70}")

    # Load this case's dataset -- overwrites the global FOOD_DATA and MEAL_FLAGS
    FOOD_DATA, MEAL_FLAGS = load_food_data(csv_path)

    case_results = {}
    case_times   = {}

    for target in EXP_TARGETS:
        print(f"\n  -- Target {target} kcal --")
        TARGET_KCAL = target

        # ---- Step 1: Run GA ------------------------------------------------
        ga_fitness_list, ga_runtime_list = [], []
        print(f"    {'Run':<5} {'StopGen':<10} {'GA Fit':<12} {'Time (s)':<10}")

        for run in range(1, N_RUNS + 1):
            t0 = time.time()
            ga_fit, stop_gen = run_simulation(
                target, pop_size=POP_SIZE, cxpb=CXPB, mutpb=MUTPB, patience=PATIENCE
            )
            rt = time.time() - t0
            ga_fitness_list.append(ga_fit)
            ga_runtime_list.append(rt)
            print(f"    {run:<5} {stop_gen:<10} {ga_fit:<12.6f} {rt:<10.4f}")

        avg_ga_time        = float(np.mean(ga_runtime_list))
        case_times[target] = avg_ga_time
        ga_stats           = _stats(ga_fitness_list)
        print(f"    GA  avg={ga_stats['mean']:.6f}  max={ga_stats['max']:.6f}  "
              f"min={ga_stats['min']:.6f}  std={ga_stats['std']:.6f}  "
              f"avg_time={avg_ga_time:.4f}s")

        # ---- Step 2: Run baselines with equal time budget ------------------
        rs_fitness_list, ls_fitness_list, sa_fitness_list = [], [], []
        budget = avg_ga_time  # all baselines get exactly the same time as the GA averaged
        print(f"\n    Time budget for baselines = {budget:.4f}s  (= GA average runtime)")
        print(f"    {'Run':<5} {'RS Fit':<12} {'LS Fit':<12} {'SA Fit':<12}")

        for run in range(1, N_RUNS + 1):
            _, rs_fit, _, _ = run_random_search(budget)
            _, ls_fit, _    = run_local_search(toolbox, POP_SIZE, budget)
            _, sa_fit, _    = run_simulated_annealing(toolbox, POP_SIZE, time_limit=budget)
            rs_fitness_list.append(rs_fit)
            ls_fitness_list.append(ls_fit)
            sa_fitness_list.append(sa_fit)
            print(f"    {run:<5} {rs_fit:<12.6f} {ls_fit:<12.6f} {sa_fit:<12.6f}")

        # Print per-algorithm summary stats for the baselines
        for algo, lst in [('RS', rs_fitness_list), ('LS', ls_fitness_list), ('SA', sa_fitness_list)]:
            s = _stats(lst)
            print(f"    {algo}  avg={s['mean']:.6f}  max={s['max']:.6f}  "
                  f"min={s['min']:.6f}  std={s['std']:.6f}")

        # ---- Step 3: Store all results for this target ---------------------
        case_results[target] = {
            'GA': {'fitness': ga_fitness_list, 'stats': ga_stats, 'avg_time': avg_ga_time},
            'RS': {'fitness': rs_fitness_list, 'stats': _stats(rs_fitness_list)},
            'LS': {'fitness': ls_fitness_list, 'stats': _stats(ls_fitness_list)},
            'SA': {'fitness': sa_fitness_list, 'stats': _stats(sa_fitness_list)},
        }

    all_case_results[case_label] = case_results
    all_case_times[case_label]   = case_times
    print(f"\n  Done -- results stored under key: '{case_label}'")


print("Shared setup ready. Run any Case cell below.")




## 8. Running the Six Dataset Cases

Each cell runs one case independently. They can be executed in any order -- results accumulate in `all_case_results` incrementally.

> **Runtime estimate:** Each case takes approximately 10-20 minutes (4 algorithms x 3 targets x 10 runs).

---

### Case 1 -- 1,000 Rows (Original + Synthetic)

The original dataset augmented with **synthetically generated** food items to reach 1,000 rows. Synthetic items introduce genuine new variety into the search space.


In [ ]:
# ── Case 1: Case 1 - 1000 rows (Original + Synthetic) ──────────────────────────────────────────────────
run_case(
    case_label = 'Case 1 - 1000 rows (Original + Synthetic)',
    csv_path   = '..\\Datasets\\case1_1000_orig_synth.csv'
)


In [ ]:
# ── Case 2: Case 2 - 1000 rows (Original + Duplicates) ──────────────────────────────────────────────────
run_case(
    case_label = 'Case 2 - 1000 rows (Original + Duplicates)',
    csv_path   = '..\\Datasets\\case2_1000_orig_dup.csv'
)


In [ ]:
# ── Case 3: Case 3 - 5000 rows (Original + Synthetic) ──────────────────────────────────────────────────
run_case(
    case_label = 'Case 3 - 5000 rows (Original + Synthetic)',
    csv_path   = '..\\Datasets\\case3_5000_orig_synth.csv'
)


In [ ]:
# ── Case 4: Case 4 - 5000 rows (Original + Duplicates) ──────────────────────────────────────────────────
run_case(
    case_label = 'Case 4 - 5000 rows (Original + Duplicates)',
    csv_path   = '..\\Datasets\\case4_5000_orig_dup.csv'
)


In [ ]:
# ── Case 5: Case 5 - 1000 rows (Original + Synthetic + Dupl.) ──────────────────────────────────────────────────
run_case(
    case_label = 'Case 5 - 1000 rows (Original + Synthetic + Dupl.)',
    csv_path   = '..\\Datasets\\case5_1000_orig_synth_dup.csv'
)


In [ ]:
# ── Case 6: Case 6 - 5000 rows (Original + Synthetic + Dupl.) ──────────────────────────────────────────────────
run_case(
    case_label = 'Case 6 - 5000 rows (Original + Synthetic + Dupl.)',
    csv_path   = '..\\Datasets\\case6_5000_orig_synth_dup.csv'
)




## 9. Combined Report and Visualisation

Run this cell after completing all case cells (or any subset). It automatically adapts to however many cases have been run and produces:

1. **Printed summary table** -- mean, max, min, std per (case, target, algorithm)
2. **Box plots** -- fitness distribution across 10 runs per (case x target x algorithm)
3. **Grouped bar chart** -- mean fitness per case and algorithm per calorie target
4. **GA runtime bar chart** -- how dataset size affects computational cost
5. **Exported CSV** -- `multi_case_results_summary.csv` for external analysis


In [ ]:
if not all_case_results:
    print('No results yet — run at least one Case cell first.')
else:
    available = list(all_case_results.keys())
    n_cases   = len(available)
    n_targets = len(EXP_TARGETS)
    print(f'Building report for {n_cases} case(s) ...\n')

    # ── 1. Printed summary table ──────────────────────────────────────────────
    print('\n' + '=' * 82)
    print('   SUMMARY REPORT')
    print('=' * 82)
    for case_label in available:
        cr = all_case_results[case_label]
        print(f"\n{'─'*82}")
        print(f'  {case_label}')
        print(f"{'─'*82}")
        hdr = f"  {'Target':<10} {'Algo':<6} {'Mean':>10} {'Max':>10} {'Min':>10} {'Std':>10}  {'GA Time':>9}"
        print(hdr)
        print('  ' + '-' * (len(hdr) - 2))
        for target in EXP_TARGETS:
            if target not in cr:
                continue
            for algo in ALGOS:
                s     = cr[target][algo]['stats']
                t_str = f"{cr[target]['GA']['avg_time']:.3f}s" if algo == 'GA' else ''
                print(f"  {str(target)+' kcal':<10} {algo:<6}"
                      f" {s['mean']:>10.6f} {s['max']:>10.6f} {s['min']:>10.6f} {s['std']:>10.6f}"
                      f"  {t_str:>9}")
            print()

    # ── 2. Box plots: one row per case, one column per target ─────────────────
    fig, axes = plt.subplots(n_cases, n_targets,
                              figsize=(5 * n_targets, 4 * n_cases), squeeze=False)
    for ri, case_label in enumerate(available):
        short = case_label.split('(')[0].strip()
        for ci, target in enumerate(EXP_TARGETS):
            ax   = axes[ri][ci]
            data = [all_case_results[case_label][target][a]['fitness'] for a in ALGOS]
            bp   = ax.boxplot(data, labels=ALGOS, patch_artist=True)
            for patch, algo in zip(bp['boxes'], ALGOS):
                patch.set_facecolor(ALGO_COLORS[algo])
                patch.set_alpha(0.75)
            ax.set_title(f'{short}  |  {target} kcal', fontsize=8, fontweight='bold')
            ax.set_ylabel('Fitness', fontsize=7)
            ax.grid(True, alpha=0.3)
            ax.tick_params(labelsize=7)
    fig.suptitle(
        f'Fitness Distribution — GA vs RS vs LS vs SA\n{n_cases} Case(s) × 3 Calorie Targets',
        fontsize=13, fontweight='bold', y=1.002
    )
    plt.tight_layout()
    plt.savefig('multi_case_boxplots.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved: multi_case_boxplots.png')

    # ── 3. Grouped bar chart: mean fitness by case, per target ────────────────
    case_short = [lbl.split('(')[1].rstrip(')') if '(' in lbl else lbl for lbl in available]
    x  = np.arange(n_cases)
    bw = 0.18
    fig2, axes2 = plt.subplots(1, n_targets, figsize=(6 * n_targets, 5), squeeze=False)
    for ci, target in enumerate(EXP_TARGETS):
        ax = axes2[0][ci]
        for ai, algo in enumerate(ALGOS):
            means = [all_case_results[lbl][target][algo]['stats']['mean'] for lbl in available]
            ax.bar(x + ai * bw, means, bw, label=algo,
                   color=ALGO_COLORS[algo], alpha=0.85, edgecolor='white')
        ax.set_title(f'Mean Fitness — {target} kcal', fontsize=11, fontweight='bold')
        ax.set_xlabel('Dataset Case', fontsize=9)
        ax.set_ylabel('Mean Fitness (10 runs)', fontsize=9)
        ax.set_xticks(x + bw * 1.5)
        ax.set_xticklabels(case_short, fontsize=7, rotation=15, ha='right')
        ax.legend(fontsize=8)
        ax.grid(axis='y', alpha=0.3)
    fig2.suptitle('Mean Fitness per Dataset Case and Algorithm', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('multi_case_bar_means.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved: multi_case_bar_means.png')

    # ── 4. GA runtime bar chart ───────────────────────────────────────────────
    fig3, ax3 = plt.subplots(figsize=(max(8, n_cases * 2), 4))
    bw3 = 0.25
    x3  = np.arange(n_cases)
    for ci, target in enumerate(EXP_TARGETS):
        runtimes = [all_case_times[lbl][target] for lbl in available]
        ax3.bar(x3 + ci * bw3, runtimes, bw3, label=f'{target} kcal', alpha=0.85, edgecolor='white')
    ax3.set_title('Average GA Runtime per Dataset Case', fontsize=12, fontweight='bold')
    ax3.set_xlabel('Dataset Case', fontsize=9)
    ax3.set_ylabel('Avg. Runtime (seconds)', fontsize=9)
    ax3.set_xticks(x3 + bw3)
    ax3.set_xticklabels(case_short, fontsize=7, rotation=15, ha='right')
    ax3.legend(fontsize=8)
    ax3.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('multi_case_ga_runtimes.png', dpi=200, bbox_inches='tight')
    plt.show()
    print('Saved: multi_case_ga_runtimes.png')

    # ── 5. Export flat CSV summary ────────────────────────────────────────────
    rows = []
    for case_label in available:
        for target in EXP_TARGETS:
            if target not in all_case_results[case_label]:
                continue
            for algo in ALGOS:
                s = all_case_results[case_label][target][algo]['stats']
                rows.append({
                    'Case':             case_label,
                    'Target_kcal':      target,
                    'Algorithm':        algo,
                    'Mean_Fitness':     round(s['mean'], 6),
                    'Max_Fitness':      round(s['max'],  6),
                    'Min_Fitness':      round(s['min'],  6),
                    'Std_Fitness':      round(s['std'],  6),
                    'Avg_GA_Runtime_s': round(all_case_results[case_label][target]['GA']['avg_time'], 4)
                                        if algo == 'GA' else '',
                })
    results_df = pd.DataFrame(rows)
    results_df.to_csv('multi_case_results_summary.csv', index=False)
    print('\nSaved: multi_case_results_summary.csv')
    print(results_df.to_string(index=False))
    print('\nReport complete.')




## 10. Statistical Analysis -- Wilcoxon ests



In [ ]:
from scipy.stats import wilcoxon

print(f"{'Case':<50} {'Target':<10} {'Comparison':<15} {'Statistic':<15} {'p-value':<15} {'Significant?':<15}")
print("-" * 120)

for case_label, case_results in all_case_results.items():
    for target in EXP_TARGETS:
        comparisons = [
            ('GA vs RS', case_results[target]['GA']['fitness'], case_results[target]['RS']['fitness']),
            ('GA vs LS', case_results[target]['GA']['fitness'], case_results[target]['LS']['fitness']),
            ('GA vs SA', case_results[target]['GA']['fitness'], case_results[target]['SA']['fitness']),
            ('RS vs LS', case_results[target]['RS']['fitness'], case_results[target]['LS']['fitness']),
            ('RS vs SA', case_results[target]['RS']['fitness'], case_results[target]['SA']['fitness']),
            ('LS vs SA', case_results[target]['LS']['fitness'], case_results[target]['SA']['fitness']),
        ]

        print(f"\n========== {case_label} | Target {target} kcal ==========")
        for name, a, b in comparisons:
            try:
                stat, p = wilcoxon(a, b)
                sig = "Yes" if p < 0.05 else "No"
                print(f"{name:<15} {stat:<15.4f} {p:<15.4f} {sig:<15}")
            except ValueError as e:
                print(f"{name:<15} Skipped ({e})")
        print("-" * 75)
